# 📦 ACE-Net High-Speed Dataset Archiver (Training-Ready Zips)
### Creates 3 Modular, Training-Ready Zip Archives (`train_features.zip`, `val_features.zip`, `test_features.zip`)

### 🌟 Bakit Ganito ang Setup?
1. **Modular Zipping (Hiwalay ang `train`, `val`, at `test`):** Hindi magiging sobrang laki ang iisang zip file kaya madaling i-handle at i-load.
2. **Local SSD Buffer First:** Mag-zi-zip muna sa ultra-fast NVMe local SSD ng Colab (`/content/temp_zips/`) bago kopyahin sa Google Drive. **Hindi ito mag-h-hang o mag-d-disconnect!**
3. **Zero-Latency Training:** Sa training phase, i-u-unzip lamang ito sa local `/content/data/` sa loob ng **~20 seconds lang** bago mag-train ang PyTorch GPU!

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
import os, sys

drive.mount('/content/drive')
print('✅ Google Drive mounted successfully!')

## Step 2: High-Speed Archiving (Local SSD Buffer -> Single Drive Copy)

In [ ]:
import os, time, shutil
from pathlib import Path

# 1. Locate Directories
DRIVE_BASE = Path('/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline_training')
SOURCE_ROOT = DRIVE_BASE / 'Baseline preprocessed'
DEST_ZIP_DIR = DRIVE_BASE / 'dataset_zips'
DEST_ZIP_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_BUFFER = Path('/content/temp_zips')
LOCAL_BUFFER.mkdir(parents=True, exist_ok=True)

print("=" * 75)
print("         📦 ACE-NET MODULAR HIGH-SPEED DATASET ARCHIVER")
print(f"  Source Preprocessed Root : {SOURCE_ROOT}")
print(f"  Target Google Drive Dir  : {DEST_ZIP_DIR}")
print("=" * 75)

SPLITS_TO_ZIP = [
    {'name': 'VAL',   'source_dir': SOURCE_ROOT / 'VAL',   'zip_name': 'val_features.zip'},
    {'name': 'TEST',  'source_dir': SOURCE_ROOT / 'TEST',  'zip_name': 'test_features.zip'},
    {'name': 'TRAIN', 'source_dir': SOURCE_ROOT / 'TRAIN', 'zip_name': 'train_features.zip'},
]

grand_start = time.time()

for item in SPLITS_TO_ZIP:
    s_name = item['name']
    s_dir = item['source_dir']
    zip_filename = item['zip_name']
    
    local_zip_path = LOCAL_BUFFER / zip_filename
    final_drive_path = DEST_ZIP_DIR / zip_filename
    
    print(f"\n🚀 [1/2] Fast Zipping [{s_name} SET] on Colab Local NVMe SSD...")
    print(f"    From : {s_dir}")
    print(f"    To   : {local_zip_path}")
    
    if not s_dir.exists():
        print(f"    ⚠️ [Warning] Folder not found: {s_dir}. Skipping...")
        continue
        
    split_start = time.time()
    
    # Zip directly using native Linux zip with fast compression (-1)
    !cd "{s_dir}" && zip -q -r -1 "{local_zip_path}" .
    
    if not local_zip_path.exists():
        print(f"    ❌ Failed to create zip for {s_name}.")
        continue
        
    zip_size_mb = local_zip_path.stat().st_size / (1024**2)
    zip_size_gb = zip_size_mb / 1024
    elapsed_zip = time.time() - split_start
    print(f"    ✅ Local Zip Done: {zip_size_gb:.2f} GB ({zip_size_mb:.1f} MB) in {elapsed_zip:.1f}s")
    
    # High-speed single file stream copy to Google Drive
    print(f"🚀 [2/2] Copying {zip_filename} to Google Drive ({final_drive_path})...")
    copy_start = time.time()
    shutil.copy2(str(local_zip_path), str(final_drive_path))
    elapsed_copy = time.time() - copy_start
    print(f"    ✅ Copy to Drive Complete in {elapsed_copy:.1f}s!")
    
    # Remove local buffer file to save Colab SSD space
    local_zip_path.unlink()

total_elapsed = time.time() - grand_start
print("\n" + "=" * 75)
print("                 🎉 ALL DATASET ZIPS SUCCESSFULLY CREATED! 🎉")
print("=" * 75)
print(f"⏱️ Total Processing Time : {total_elapsed/60:.2f} minuto")
print(f"📍 Saved Google Drive Location: {DEST_ZIP_DIR}/")
print("   1. val_features.zip")
print("   2. test_features.zip")
print("   3. train_features.zip")
print("=" * 75)